In [ ]:
# ---- 0. Setup & Konfigurasi ----
import os, sys, json, time, warnings, random, pickle, zipfile, platform, itertools, hashlib
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
if "google.colab" not in sys.modules:
    matplotlib.use(os.environ.get("MPLBACKEND", "Agg"))
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------ KONFIGURASI ------------------------------
FAST_MODE       = False                         # <- ubah ke False untuk run sebenarnya
DATA_PATH       = "retail_pricing_demand_100k.csv"
SEED            = 42
TRAIN_RATIO     = 0.80
LAG_DAYS        = [1, 7, 14, 30]
ROLLING_WINDOWS = [7, 14, 30]
FEATURE_SET     = "strict"                     # 'strict' = hanya informasi yang diketahui sebelum waktu prediksi | 'original' = 44 fitur lama
COMPARISON      = "none"                       # "full" | "simple" | "none"
ROLL_BLOCK      = 7                            # panjang tiap blok uji rolling-origin (hari)
N_FOLDS         = 3 if FAST_MODE else 5

RF_ITER         = 3  if FAST_MODE else 20
LGB_ITER        = 3  if FAST_MODE else 20
LSTM_ITER       = 2  if FAST_MODE else 10
CV_SPLITS       = 2  if FAST_MODE else 3
LSTM_EPOCHS     = 3  if FAST_MODE else 100
LSTM_PATIENCE   = 2  if FAST_MODE else 15
LSTM_SEEDS      = [42] if FAST_MODE else [42, 43, 44]
ABL_TREES       = 30 if FAST_MODE else 200
PERM_REPEATS    = 2  if FAST_MODE else 5

# Asumsi simulasi inventori.
HOLD_ANNUAL     = 0.25                         # biaya simpan per tahun (fraksi harga)
MARGIN          = 0.30                         # biaya lost-sale per unit (fraksi harga)
ORDER_COST      = 50.0                         # biaya tetap per pesanan
BASE_L, BASE_R, BASE_Z = 7, 7, 1.65            # lead time, review period (hari), z-score service level

DATA_DIR, RES_DIR, FIG_DIR = "data", "results", "figures"
for _d in (DATA_DIR, RES_DIR, FIG_DIR, f"{RES_DIR}/cache"):
    os.makedirs(_d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
T_START = time.time()

BASELINES = ["Naive", "SeasonalNaive7", "MA7", "MA30", "HistMean"]
def model_list():
    m = []
    if COMPARISON in ("simple", "full"): m += BASELINES
    if COMPARISON == "full": m += ["Ridge", "LightGBM"]
    return m + ["RF", "LSTM"]

_FP = hashlib.md5(json.dumps([FEATURE_SET, SEED, TRAIN_RATIO, LAG_DAYS, ROLLING_WINDOWS, RF_ITER, LGB_ITER, LSTM_ITER,
                              CV_SPLITS, LSTM_EPOCHS, LSTM_PATIENCE, FAST_MODE]).encode()).hexdigest()[:6]
def cached(name, fn):
    """Cache hasil tahap berat; otomatis tidak dipakai kalau konfigurasi berubah."""
    p = f"{RES_DIR}/cache/{name}_{_FP}.pkl"
    if os.path.exists(p):
        print(f"[cache] '{name}' dimuat dari {p}"); return pickle.load(open(p, "rb"))
    t0 = time.time(); r = fn(); pickle.dump(r, open(p, "wb"))
    print(f"[selesai] '{name}' dalam {time.time()-t0:.0f} detik"); return r

def metrics(y, p, scale=None):
    e = y - p; ae = np.abs(e); pos = y > 0
    o = dict(MAE=ae.mean(), RMSE=float(np.sqrt((e ** 2).mean())), MAPE=float(np.mean(ae[pos] / y[pos]) * 100),
             WAPE=float(ae.sum() / y.sum() * 100), sMAPE=float(np.mean(2 * ae / (np.abs(y) + np.abs(p) + 1e-9)) * 100))
    if scale is not None: o["MASE"] = float(np.mean(ae / scale))
    return o

def load_panel():
    """Memuat data/panel.csv + metadata dan menyiapkan variabel global yang dipakai tiap section."""
    global D, META, P, T, TT, Y, Yflat, split_idx, starts, N_FOLDS, FEATS, GROUPS, Xmain, tr_h, te_h
    D = pd.read_csv(f"{DATA_DIR}/panel.csv", parse_dates=["date"])
    META = json.load(open(f"{DATA_DIR}/metadata.json"))
    P, T = META["n_products"], META["n_days_usable"]
    assert len(D) == P * T and (D["t"].values.reshape(P, T) == np.arange(T)).all(), "panel.csv tidak konsisten - jalankan ulang Preprocessing"
    TT = D["t"].values
    Y = D["units_sold"].values.reshape(P, T).astype(float); Yflat = Y.reshape(-1)
    split_idx, starts = META["split_day_index"], META["rolling_block_starts"]; N_FOLDS = len(starts)
    FEATS, GROUPS = META["feature_sets"][META["main_set"]], META["groups_main"]
    Xmain = D[FEATS].values.astype(np.float32)
    tr_h, te_h = np.where(TT < split_idx)[0], np.where(TT >= split_idx)[0]
    print(f"Panel dimuat: {P} produk x {T} hari | fitur ({META['main_set']}): {len(FEATS)} | train hari 0-{split_idx-1}, test hari {split_idx}-{T-1} | {N_FOLDS} blok rolling-origin")

def prod_scale(train_end):
    s = np.abs(np.diff(Y[:, :train_end], axis=1)).mean(1)
    return np.repeat(np.maximum(s, 1e-6), T)

def make_cv(tt_train, n_splits):
    from sklearn.model_selection import TimeSeriesSplit
    days = np.unique(tt_train)
    return [(np.where(np.isin(tt_train, days[a]))[0], np.where(np.isin(tt_train, days[b]))[0])
            for a, b in TimeSeriesSplit(n_splits=n_splits).split(days)]

def tune_sklearn(est, space, n_iter, name):
    """RandomizedSearchCV dengan validasi berurutan waktu dan HANYA data latih."""
    from sklearn.model_selection import RandomizedSearchCV
    t0 = time.time()
    s = RandomizedSearchCV(est, space, n_iter=n_iter, cv=make_cv(TT[tr_h], CV_SPLITS), scoring="neg_mean_absolute_error",
                           random_state=SEED, n_jobs=1, refit=False)
    s.fit(Xmain[tr_h], Yflat[tr_h])
    print(f"{name}: CV MAE terbaik = {-s.best_score_:.4f} ({time.time()-t0:.0f} dtk) | {s.best_params_}")
    return {"best_params": s.best_params_, "best_cv_mae": float(-s.best_score_),
            "table": pd.DataFrame(s.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]}

def save_pred(name, **arrays):
    np.savez_compressed(f"{RES_DIR}/pred_{name}.npz", **arrays)

def load_all_preds():
    z = np.load(f"{RES_DIR}/predictions_all.npz"); PH, PR = {}, {}
    for k in z.files:
        kind, m = k.split("__", 1); (PH if kind == "holdout" else PR)[m] = z[k]
    return PH, PR

print(f"Konfigurasi: FAST_MODE={FAST_MODE} | FEATURE_SET={FEATURE_SET} | COMPARISON={COMPARISON} | model: {model_list()}")

## 1. Preprocessing

Memuat dataset, memeriksa kualitas data, mengagregasi data pada tingkat tanggal dan produk, membentuk fitur tanpa kebocoran informasi, menentukan pembagian data, serta menyimpan hasil preprocessing ke folder `data/`.

In [ ]:
# ---- 1.1 Memuat dataset ----
print("\n[1/7] Memuat dataset...")
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Upload file retail_pricing_demand_100k.csv ...")
        DATA_PATH = list(files.upload().keys())[0]
    except Exception:
        raise FileNotFoundError(f"{DATA_PATH} tidak ditemukan")
raw = pd.read_csv(DATA_PATH)
raw["date"] = pd.to_datetime(raw["date"])
print(f"      Raw shape       : {raw.shape}")
print(f"      Rentang tanggal : {raw['date'].min().date()} s/d {raw['date'].max().date()}")
print(f"      Produk unik     : {raw['product_id'].nunique()}")
print(f"      Kategori        : {raw['category'].nunique()} | Region: {raw['region'].nunique()} | Channel: {raw['channel'].nunique() if 'channel' in raw else '-'}")

In [ ]:
# ---- 1.2 Missing values dan audit data ----
print("\n[2/7] Missing values dan audit data...")
miss = raw.isnull().sum(); miss = miss[miss > 0]
print("      Kolom dengan missing:", miss.to_dict())
raw["promotion_type"] = raw["promotion_type"].fillna("None")      # null berarti tidak ada promosi

AUDIT = {"raw_rows": len(raw), "n_products": int(raw["product_id"].nunique()),
         "date_min": str(raw["date"].min().date()), "date_max": str(raw["date"].max().date()), "n_days": int(raw["date"].nunique())}
_g = raw.groupby(["date", "product_id"])
AUDIT["rows_per_product_day"] = {int(k): int(v) for k, v in _g.size().value_counts().items()}
AUDIT["regions_per_product"] = {int(k): int(v) for k, v in raw.groupby("product_id")["region"].nunique().value_counts().items()}
AUDIT["regions_per_product_day"] = {int(k): int(v) for k, v in _g["region"].nunique().value_counts().items()}
AUDIT["categories_per_product"] = {int(k): int(v) for k, v in raw.groupby("product_id")["category"].nunique().value_counts().items()}
if "channel" in raw:
    AUDIT["channels_per_product_day"] = {int(k): int(v) for k, v in _g["channel"].nunique().value_counts().items()}
AUDIT["promo_share_rows_%"] = round(float((raw["promotion_type"] != "None").mean() * 100), 2)
AUDIT["units_sold_describe_raw_rows"] = raw["units_sold"].describe().round(3).to_dict()
AUDIT["n_zero_units_raw_rows"] = int((raw["units_sold"] <= 0).sum())
AUDIT["stockout_share_%"] = round(float(raw["stockout_flag"].mean() * 100), 2)
json.dump(AUDIT, open(f"{RES_DIR}/audit_summary.json", "w"), indent=1, default=str)
print(json.dumps(AUDIT, indent=1, default=str))

_promo = raw.groupby("promotion_type")["units_sold"].agg(["count", "mean", "std"]).round(2)
_dow = raw.groupby(raw["date"].dt.dayofweek)["units_sold"].mean().round(2).rename("mean_units")
_cat = raw.groupby("category")["units_sold"].agg(["mean", "std"]).round(2)
_month = raw.groupby([raw["date"].dt.month, "season"])["units_sold"].agg(["count", "mean"]).round(2)
_num_cols = [c for c in ["units_sold", "current_price", "discount_pct", "price_change_pct", "inventory_level", "stockout_flag", "demand_index"] if c in raw]
_corr = raw[_num_cols].corr()["units_sold"].round(3)
_daily = raw.groupby("date")["units_sold"].mean()
print("\nRata-rata units per tipe promosi:\n", _promo); print("\nRata-rata units per bulan/musim:\n", _month)
print("\nKorelasi hari-yang-sama dengan units_sold:\n", _corr)
print("\nPerubahan terbesar rata-rata harian (indikasi level shift):\n", _daily.diff().abs().sort_values(ascending=False).head(3).round(2))
for nm, t in [("audit_promotion", _promo), ("audit_dayofweek", _dow.to_frame()), ("audit_category", _cat),
              ("audit_month_season", _month), ("audit_corr_units", _corr.to_frame())]:
    t.to_csv(f"{RES_DIR}/{nm}.csv")

In [ ]:
# ---- 1.3 Agregasi ke level tanggal + produk ----
print("\n[3/7] Mengagregasi data ke level date + product_id...")
raw["_has_promo"] = (raw["promotion_type"] != "None").astype(int)
raw = raw.sort_values(["date", "product_id", "_has_promo"], ascending=[True, True, False])   # baris berpromosi didahulukan
agg = raw.groupby(["date", "product_id"], sort=False).agg(
    units_sold=("units_sold", "sum"), revenue=("revenue", "sum"),
    inventory_level=("inventory_level", "mean"), current_price=("current_price", "mean"),
    base_price=("base_price", "mean"), discount_pct=("discount_pct", "mean"),
    price_change_pct=("price_change_pct", "mean"), stockout_flag=("stockout_flag", "max"),
    demand_index=("demand_index", "mean"), promo_share=("_has_promo", "mean"),
    category=("category", "first"), brand=("brand", "first"), season=("season", "first"),
    promotion_type=("promotion_type", "first")).reset_index()          # 'region' sengaja tidak dipakai: 1 produk-hari memuat beberapa region
d = agg.sort_values(["product_id", "date"]).reset_index(drop=True)
P_ = d["product_id"].nunique(); T_all = d["date"].nunique()
assert len(d) == P_ * T_all, "panel produk x tanggal tidak lengkap"
print(f"      Shape setelah agregasi : {d.shape} ({P_} produk x {T_all} hari)")
print(f"      units_sold (level produk-hari): mean {d['units_sold'].mean():.2f} ± {d['units_sold'].std():.2f} | min {d['units_sold'].min()} | max {d['units_sold'].max()}")

In [ ]:
# ---- 1.4 Feature engineering (semua lag/rolling hanya memakai data sampai t-1) ----
print("\n[4/7] Feature engineering...")
def wide_of(col): return pd.DataFrame(d[col].values.reshape(P_, T_all).T)       # (T, P)
def flat(w): return w.values.T.reshape(-1)

d["day_of_week"] = d["date"].dt.dayofweek; d["day_of_month"] = d["date"].dt.day
d["week_of_year"] = d["date"].dt.isocalendar().week.astype(int).values
d["month"] = d["date"].dt.month; d["quarter"] = d["date"].dt.quarter
d["is_weekend"] = (d["day_of_week"] >= 5).astype(int)

W = wide_of("units_sold")
for lag in LAG_DAYS:
    d[f"lag_{lag}"] = flat(W.shift(lag))
S1 = W.shift(1)                                    # shift(1) SEBELUM window -> tidak ada informasi hari target
for w in ROLLING_WINDOWS:
    r = S1.rolling(w, min_periods=1)
    d[f"rolling_mean_{w}d"] = flat(r.mean()); d[f"rolling_std_{w}d"] = flat(r.std().fillna(0)); d[f"rolling_max_{w}d"] = flat(r.max())

d["price_ratio"] = d["current_price"] / (d["base_price"] + 1e-6)
d["discount_flag"] = (d["discount_pct"] > 0).astype(int)
d["has_promotion"] = (d["promotion_type"] != "None").astype(int)
d["effective_price"] = d["current_price"] * (1 - d["discount_pct"] / 100)
# sinyal inventori/permintaan: versi hari-yang-sama (original) dan versi lag-1 (strict)
d["inventory_lag1"] = flat(wide_of("inventory_level").shift(1))
d["stockout_lag1"] = flat(wide_of("stockout_flag").shift(1))
d["demand_index_lag1"] = flat(wide_of("demand_index").shift(1))
d["inventory_ratio"] = d["inventory_level"] / (d["rolling_mean_7d"] + 1e-6)
d["days_of_supply"] = d["inventory_level"] / (d["rolling_mean_7d"].clip(lower=0.1) + 1e-6)
d["inventory_ratio_lag1"] = d["inventory_lag1"] / (d["rolling_mean_7d"] + 1e-6)
d["days_of_supply_lag1"] = d["inventory_lag1"] / (d["rolling_mean_7d"].clip(lower=0.1) + 1e-6)
print(f"      Fitur numerik dibuat. Kolom saat ini: {d.shape[1]}")

In [ ]:
# ---- 1.5 Encoding fitur kategorikal ----
print("\n[5/7] Encoding fitur kategorikal...")
label_maps = {}
for c in ["category", "brand", "season", "promotion_type"]:
    cats = sorted(d[c].dropna().unique().tolist())
    label_maps[c] = {k: i for i, k in enumerate(cats)}
    d[f"{c}_enc"] = d[c].map(label_maps[c])
    print(f"      {c:15s}: {len(cats)} kategori -> {c}_enc")
PROMO_COLS = []
for k in label_maps["promotion_type"]:
    d[f"promo_{k}"] = (d["promotion_type"] == k).astype(int); PROMO_COLS.append(f"promo_{k}")
json.dump(label_maps, open(f"{DATA_DIR}/label_maps.json", "w"), indent=2)
print("      One-hot promotion_type:", PROMO_COLS)

In [ ]:
# ---- 1.6 Feature set dan definisi split ----
print("\n[6/7] Mendefinisikan feature set dan split waktu...")
GROUPS_ORIG = {
    "price": ["current_price", "base_price", "price_change_pct", "discount_pct", "price_ratio", "discount_flag", "effective_price"],
    "promotion": ["has_promotion", "promotion_type_enc"] + PROMO_COLS,
    "temporal": ["day_of_week", "day_of_month", "week_of_year", "month", "quarter", "is_weekend"],
    "lag": [f"lag_{x}" for x in LAG_DAYS],
    "rolling": [f"rolling_{s}_{w}d" for s in ["mean", "std", "max"] for w in ROLLING_WINDOWS],
    "inventory": ["inventory_level", "inventory_lag1", "inventory_ratio", "days_of_supply", "stockout_flag"],
    "context": ["category_enc", "brand_enc", "season_enc", "demand_index"],
}
GROUPS_STRICT = dict(GROUPS_ORIG)
GROUPS_STRICT["promotion"] = GROUPS_ORIG["promotion"] + ["promo_share"]
GROUPS_STRICT["inventory"] = ["inventory_lag1", "inventory_ratio_lag1", "days_of_supply_lag1", "stockout_lag1"]
GROUPS_STRICT["context"] = ["category_enc", "brand_enc", "season_enc", "demand_index_lag1"]

ALL_NEEDED = sorted({c for g in (GROUPS_ORIG, GROUPS_STRICT) for v in g.values() for c in v})
Dp = d.dropna(subset=ALL_NEEDED + ["units_sold"]).copy()
dates_u = np.sort(Dp["date"].unique()); T_use = len(dates_u)
Dp["t"] = Dp["date"].map({x: i for i, x in enumerate(dates_u)}).astype(int)
Dp = Dp.sort_values(["product_id", "t"]).reset_index(drop=True)
assert (Dp.groupby("product_id").size() == T_use).all()
const_cols = [c for c in ALL_NEEDED if Dp[c].nunique() <= 1]
print(f"      Baris dihapus karena warm-up lag/rolling: {len(d)-len(Dp):,} | baris siap pakai: {len(Dp):,} ({T_use} hari)")
print(f"      Kolom konstan yang dibuang dari fitur  : {const_cols}")
for g in (GROUPS_ORIG, GROUPS_STRICT):
    for k in g: g[k] = [c for c in g[k] if c not in const_cols]
FS = {"original": [c for v in GROUPS_ORIG.values() for c in v], "strict": [c for v in GROUPS_STRICT.values() for c in v]}
split_day = int(T_use * TRAIN_RATIO)
roll_starts = [T_use - (N_FOLDS - k) * ROLL_BLOCK for k in range(N_FOLDS)]
print(f"      Jumlah fitur: { {k: len(v) for k, v in FS.items()} } | fitur utama: {FEATURE_SET}")
print(f"      Holdout 80:20 -> train hari 0-{split_day-1} ({str(dates_u[0])[:10]} .. {str(dates_u[split_day-1])[:10]}), test hari {split_day}-{T_use-1} ({str(dates_u[split_day])[:10]} .. {str(dates_u[-1])[:10]})")
print(f"      Blok rolling-origin (indeks hari): {[(s, s + ROLL_BLOCK - 1) for s in roll_starts]}")

In [ ]:
# ---- 1.7 Menyimpan ke folder data/ ----
print("\n[7/7] Menyimpan hasil preprocessing...")
keep = ["date", "product_id", "category", "t", "units_sold"] + [c for c in ALL_NEEDED if c not in ("units_sold",)]
Dp[list(dict.fromkeys(keep))].to_csv(f"{DATA_DIR}/panel.csv", index=False)
GROUPS_MAIN = GROUPS_STRICT if FEATURE_SET == "strict" else GROUPS_ORIG
metadata = {"main_set": FEATURE_SET, "feature_sets": FS, "groups_main": GROUPS_MAIN, "groups_strict": GROUPS_STRICT, "groups_original": GROUPS_ORIG,
            "label_maps": label_maps, "target": "units_sold", "n_products": int(P_), "n_days_usable": int(T_use),
            "split_day_index": int(split_day), "split_date": str(dates_u[split_day])[:10],
            "rolling_block_starts": [int(x) for x in roll_starts], "roll_block": ROLL_BLOCK,
            "dates": [str(x)[:10] for x in dates_u], "lag_days": LAG_DAYS, "rolling_windows": ROLLING_WINDOWS, "random_seed": SEED}
json.dump(metadata, open(f"{DATA_DIR}/metadata.json", "w"), indent=1)
json.dump({"feature_sets": FS, "groups_strict": GROUPS_STRICT, "groups_original": GROUPS_ORIG}, open(f"{RES_DIR}/feature_definitions.json", "w"), indent=1)
print(f"      Tersimpan di '{DATA_DIR}/': panel.csv, metadata.json, label_maps.json  |  '{RES_DIR}/': audit_*.csv, feature_definitions.json")
print("  PREPROCESSING SELESAI")

## 2. Random Forest

Melakukan tuning menggunakan data latih dan validasi berurutan waktu, evaluasi holdout 80:20 dan rolling-origin, analisis feature importance, serta ablasi fitur.

In [ ]:
# ---- 2.0 Impor dan muat data ----
from sklearn.ensemble import RandomForestRegressor
load_panel()
RF_SPACE = {"n_estimators": [100, 200, 300, 500], "max_depth": [None, 10, 20, 30], "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4], "max_features": ["sqrt", "log2", 0.5], "bootstrap": [True, False]}
if FAST_MODE: RF_SPACE.update(n_estimators=[20, 40], max_depth=[10, 20])

In [ ]:
# ---- 2.1 Hyperparameter tuning ----
print("\n[1/5] Hyperparameter tuning Random Forest (RandomizedSearchCV, validasi berurutan waktu, data latih saja)...")
RF_T = cached("tune_rf", lambda: tune_sklearn(RandomForestRegressor(random_state=SEED, n_jobs=-1), RF_SPACE, RF_ITER, "RF"))
print("      Parameter terbaik:", RF_T["best_params"])

In [ ]:
# ---- 2.2 Evaluasi holdout 80:20 ----
print("\n[2/5] Training dan evaluasi RF pada holdout 80:20...")
def fit_rf(train_end, test_ts):
    tr = np.where(TT < train_end)[0]; te = np.where(np.isin(TT, test_ts))[0]
    m = RandomForestRegressor(random_state=SEED, n_jobs=-1, **RF_T["best_params"]).fit(Xmain[tr], Yflat[tr])
    out = np.full(len(D), np.nan); out[te] = m.predict(Xmain[te]); return m, out

t0 = time.time()
rf, rf_holdout = fit_rf(split_idx, list(range(split_idx, T)))
rf_train_time = time.time() - t0
sc_h = prod_scale(split_idx)
rf_metrics = metrics(Yflat[te_h], rf_holdout[te_h], sc_h[te_h])
rf_train_mae = float(np.abs(rf.predict(Xmain[tr_h]) - Yflat[tr_h]).mean())
print(f"      Waktu training: {rf_train_time:.0f} dtk | MAE train {rf_train_mae:.3f}")
print("      Holdout:", {k: round(v, 4) for k, v in rf_metrics.items()})

In [ ]:
# ---- 2.3 Rolling-origin ----
print("\n[3/5] Evaluasi rolling-origin (expanding window, hyperparameter tetap)...")
rf_rolling = np.full(len(D), np.nan)
for k, s in enumerate(starts):
    ts = list(range(s, s + ROLL_BLOCK)); te = np.where(np.isin(TT, ts))[0]
    _, pk = fit_rf(s, ts); rf_rolling[te] = pk[te]
    print(f"      fold {k+1}/{N_FOLDS}: latih hari <{s}, uji hari {ts[0]}-{ts[-1]}")

In [ ]:
# ---- 2.4 Feature importance (Gini dan permutation) ----
print("\n[4/5] Feature importance...")
gini = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
rng = np.random.default_rng(SEED); Xte, yte = Xmain[te_h], Yflat[te_h]
base_mae = np.abs(rf.predict(Xte) - yte).mean()
def perm(idx, n):
    inc = []
    for _ in range(n):
        Xp = Xte.copy(); Xp[:, idx] = Xte[rng.permutation(len(Xte))][:, idx]
        inc.append(np.abs(rf.predict(Xp) - yte).mean() - base_mae)
    return float(np.mean(inc)), float(np.std(inc))
pg = pd.DataFrame({g: perm([FEATS.index(c) for c in cols], PERM_REPEATS) for g, cols in GROUPS.items() if cols},
                  index=["MAE_increase", "std"]).T.sort_values("MAE_increase", ascending=False)
pf = pd.DataFrame({f: perm([FEATS.index(f)], max(2, PERM_REPEATS - 2)) for f in gini.index[:12]},
                  index=["MAE_increase", "std"]).T.sort_values("MAE_increase", ascending=False)
grp_gini = pd.Series({g: float(gini[[c for c in cols if c in gini.index]].sum()) for g, cols in GROUPS.items()}, name="gini_share")
print("      Gini per kelompok:\n", grp_gini.round(3).to_string()); print("\n      Permutation importance per kelompok (kenaikan MAE):\n", pg.round(3).to_string())

In [ ]:
# ---- 2.5 Analisis fitur: ablasi, korelasi, VIF ----
print("\n[5/5] Ablasi fitur, korelasi dan VIF fitur rolling...")
def ablation_stage():
    s, g = META["feature_sets"]["strict"], META["groups_strict"]
    roll_all = g["rolling"]; roll14_30 = [c for c in roll_all if ("14d" in c or "30d" in c)]
    sets = {"original (incl. same-day signals)": META["feature_sets"]["original"], "strict (main)": s,
            "strict - all rolling stats": [c for c in s if c not in roll_all],
            "strict - rolling 14d/30d (keep 7d)": [c for c in s if c not in roll14_30],
            "strict - lag features": [c for c in s if c not in g["lag"]],
            "strict - lagged demand_index/inventory/stockout": [c for c in s if c not in g["inventory"] + ["demand_index_lag1"]],
            "strict - price & promotion": [c for c in s if c not in g["price"] + g["promotion"]],
            "strict - rolling & lag (no autoregressive info)": [c for c in s if c not in roll_all + g["lag"]]}
    rows = []
    for nm, cols in sets.items():
        X = D[cols].values.astype(np.float32)
        m = RandomForestRegressor(random_state=SEED, n_jobs=-1, **{**RF_T["best_params"], "n_estimators": ABL_TREES}).fit(X[tr_h], Yflat[tr_h])
        rows.append({"feature_set": nm, "n_features": len(cols), **metrics(Yflat[te_h], m.predict(X[te_h]), sc_h[te_h])}); print("      ", rows[-1])
    return pd.DataFrame(rows)
ABL = cached("ablation", ablation_stage)

cols_c = META["groups_strict"]["rolling"] + META["groups_strict"]["lag"]
Xc = D.loc[D["t"] < split_idx, cols_c]; corr = Xc.corr()
def vif(X):
    A = X.values.astype(float); out = {}
    for i, c in enumerate(X.columns):
        o = np.c_[np.ones(len(A)), np.delete(A, i, axis=1)]
        b, *_ = np.linalg.lstsq(o, A[:, i], rcond=None)
        r2 = 1 - ((A[:, i] - o @ b) ** 2).sum() / ((A[:, i] - A[:, i].mean()) ** 2).sum(); out[c] = 1 / max(1 - r2, 1e-9)
    return pd.Series(out, name="VIF")
vif_s = vif(Xc).sort_values(ascending=False)
print("\n      Ablasi RF (holdout):\n", ABL.round(3).to_string(index=False)); print("\n      VIF teratas:\n", vif_s.head(6).round(1).to_string())

In [ ]:
# ---- 2.6 Simpan hasil Random Forest ----
save_pred("RF", holdout=rf_holdout, rolling=rf_rolling)
gini.rename("gini").to_csv(f"{RES_DIR}/rf_gini_importance.csv"); grp_gini.to_csv(f"{RES_DIR}/rf_group_gini.csv")
pg.to_csv(f"{RES_DIR}/rf_perm_importance_group.csv"); pf.to_csv(f"{RES_DIR}/rf_perm_importance_feature.csv")
ABL.to_csv(f"{RES_DIR}/rf_ablation.csv", index=False); corr.to_csv(f"{RES_DIR}/corr_rolling_lag.csv"); vif_s.to_csv(f"{RES_DIR}/vif_rolling_lag.csv")
RF_T["table"].to_csv(f"{RES_DIR}/rf_tuning.csv", index=False)
json.dump({"model": "Random Forest", "best_params": {k: str(v) for k, v in RF_T["best_params"].items()}, "search_space": {k: str(v) for k, v in RF_SPACE.items()},
           "n_iter": RF_ITER, "cv_splits": CV_SPLITS, "best_cv_mae": RF_T["best_cv_mae"], "train_time_sec": round(rf_train_time, 1),
           "train_mae": rf_train_mae, "holdout_metrics": rf_metrics, "n_features": len(FEATS)}, open(f"{RES_DIR}/rf_results.json", "w"), indent=1)
print("  RANDOM FOREST SELESAI -> results/pred_RF.npz, rf_*.csv/json")

## 3. LSTM

Melatih dan mengevaluasi LSTM dengan fitur yang sama seperti Random Forest. Proses scaling hanya menggunakan data latih, tuning menggunakan validasi berurutan waktu, dan eksperimen dijalankan dengan tiga seed.

In [ ]:
# ---- 3.0 Impor dan muat data ----
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
load_panel()
print("TensorFlow", tf.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")) > 0)
LSTM_SPACE = {"L": [7, 14], "units": [64, 128], "layers": [1, 2], "dropout": [0.1, 0.2, 0.3],
              "lr": [1e-3, 5e-4], "batch": [64, 128], "loss": ["mse", "huber"]}

In [ ]:
# ---- 3.1 Fungsi sequence, arsitektur dan training ----
def lstm_prep(feats, train_end):
    X = D[feats].values.astype(np.float32)
    sc = MinMaxScaler().fit(X[TT < train_end])                 # scaler hanya dari data latih
    Xs = sc.transform(X).astype(np.float32).reshape(P, T, len(feats))
    mu, sd = Y[:, :train_end].mean(), Y[:, :train_end].std()
    return Xs, (Y - mu) / sd, mu, sd

def windows(Xs, L, ts):                                        # window = baris t-L+1..t, sequence tidak melintasi batas produk
    return np.concatenate([Xs[:, t - L + 1:t + 1, :] for t in ts], axis=0)

def build_lstm(L, F, cfg):
    keras.backend.clear_session()
    m = keras.Sequential([keras.layers.Input((L, F))])
    for i in range(cfg["layers"]):
        last = i == cfg["layers"] - 1
        m.add(keras.layers.LSTM(cfg["units"] if not last or cfg["layers"] == 1 else cfg["units"] // 2, return_sequences=not last))
        m.add(keras.layers.Dropout(cfg["dropout"]))
    m.add(keras.layers.Dense(32, activation="relu")); m.add(keras.layers.Dense(1))
    m.compile(optimizer=keras.optimizers.Adam(cfg["lr"]), loss=keras.losses.Huber(1.0) if cfg["loss"] == "huber" else "mse", metrics=["mae"])
    return m

def lstm_run(cfg, train_end, seed, pred_ts=None, val_days=7):
    """Latih LSTM pada hari < train_end (7 hari terakhir = validasi berurutan waktu untuk early stopping)."""
    L = cfg["L"]; Xs, Ys, mu, sd = lstm_prep(FEATS, train_end)
    t_all = np.arange(L - 1, train_end)
    t_va, t_tr = t_all[t_all >= train_end - val_days], t_all[t_all < train_end - val_days]
    Xtr, ytr = windows(Xs, L, t_tr), Ys[:, t_tr].T.reshape(-1)
    Xva, yva = windows(Xs, L, t_va), Ys[:, t_va].T.reshape(-1)
    keras.utils.set_random_seed(seed)
    m = build_lstm(L, len(FEATS), cfg)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=LSTM_PATIENCE, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(1, LSTM_PATIENCE // 2), min_lr=1e-6)]
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=LSTM_EPOCHS, batch_size=cfg["batch"], callbacks=cb, verbose=0)
    info = {"epochs_run": len(h.history["loss"]), "best_epoch": int(np.argmin(h.history["val_loss"]) + 1), "n_params": int(m.count_params()),
            "history": {k: [float(x) for x in h.history[k]] for k in ("loss", "val_loss")}}
    if pred_ts is None:
        pv = m.predict(Xva, batch_size=1024, verbose=0).ravel() * sd + mu
        return float(np.mean(np.abs(pv - (yva * sd + mu)))), info
    pt = m.predict(windows(Xs, L, pred_ts), batch_size=1024, verbose=0).ravel() * sd + mu
    out = np.full((P, T), np.nan); out[:, pred_ts] = pt.reshape(len(pred_ts), P).T
    return out.reshape(-1), info

In [ ]:
# ---- 3.2 Hyperparameter tuning ----
print("\n[1/4] Hyperparameter tuning LSTM (random search, validasi 7 hari terakhir periode latih)...")
def tune_lstm():
    combos = list(itertools.product(*LSTM_SPACE.values()))
    picks = random.Random(SEED).sample(combos, min(LSTM_ITER, len(combos))); rows = []
    for c in picks:
        cfg = dict(zip(LSTM_SPACE.keys(), c)); t0 = time.time()
        mae, info = lstm_run(cfg, split_idx, SEED)
        rows.append({**cfg, "val_MAE": mae, "epochs_run": info["epochs_run"], "best_epoch": info["best_epoch"], "n_params": info["n_params"], "sec": round(time.time() - t0)})
        print("      ", rows[-1])
    tab = pd.DataFrame(rows).sort_values("val_MAE")
    best = {k: (v.item() if hasattr(v, "item") else v) for k, v in tab.iloc[0][list(LSTM_SPACE)].to_dict().items()}
    return {"best_cfg": best, "table": tab}
LSTM_T = cached("tune_lstm", tune_lstm)
CFG = LSTM_T["best_cfg"]; print("      Konfigurasi terbaik:", CFG)

In [ ]:
# ---- 3.3 Evaluasi holdout (beberapa seed) ----
print(f"\n[2/4] Training dan evaluasi LSTM pada holdout 80:20 (seed {LSTM_SEEDS})...")
test_ts = list(range(split_idx, T)); sc_h = prod_scale(split_idx)
lstm_hold, lstm_info, seed_rows = {}, {}, []
for sd_ in LSTM_SEEDS:
    t0 = time.time(); lstm_hold[sd_], lstm_info[sd_] = lstm_run(CFG, split_idx, sd_, pred_ts=np.array(test_ts))
    seed_rows.append({"seed": sd_, **metrics(Yflat[te_h], lstm_hold[sd_][te_h], sc_h[te_h]), "epochs_run": lstm_info[sd_]["epochs_run"],
                      "best_epoch": lstm_info[sd_]["best_epoch"], "n_params": lstm_info[sd_]["n_params"], "sec": round(time.time() - t0)})
    print("      ", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in seed_rows[-1].items()})
seed_tab = pd.DataFrame(seed_rows)

In [ ]:
# ---- 3.4 Rolling-origin ----
print("\n[3/4] Evaluasi rolling-origin LSTM (konfigurasi tetap, semua seed)...")
lstm_roll = {s_: np.full(len(D), np.nan) for s_ in LSTM_SEEDS}
for k, s in enumerate(starts):
    ts = list(range(s, s + ROLL_BLOCK)); te = np.where(np.isin(TT, ts))[0]; t0 = time.time()
    for sd_ in LSTM_SEEDS:
        pk, _ = lstm_run(CFG, s, sd_, pred_ts=np.array(ts)); lstm_roll[sd_][te] = pk[te]
    print(f"      fold {k+1}/{N_FOLDS}: latih hari <{s}, uji hari {ts[0]}-{ts[-1]} ({time.time()-t0:.0f} dtk)")

In [ ]:
# ---- 3.5 Simpan hasil LSTM ----
print("\n[4/4] Menyimpan hasil LSTM...")
main_seed = LSTM_SEEDS[0]
save_pred("LSTM", holdout=lstm_hold[main_seed], rolling=lstm_roll[main_seed],
          **{f"holdout_s{s_}": lstm_hold[s_] for s_ in LSTM_SEEDS}, **{f"rolling_s{s_}": lstm_roll[s_] for s_ in LSTM_SEEDS})
seed_tab.round(4).to_csv(f"{RES_DIR}/lstm_seed_variability.csv", index=False); LSTM_T["table"].to_csv(f"{RES_DIR}/lstm_tuning.csv", index=False)
json.dump({"model": "LSTM", "best_cfg": CFG, "search_space": {k: str(v) for k, v in LSTM_SPACE.items()}, "n_iter": LSTM_ITER, "seeds": LSTM_SEEDS,
           "epochs_max": LSTM_EPOCHS, "patience": LSTM_PATIENCE, "n_params": lstm_info[main_seed]["n_params"], "epochs_run": lstm_info[main_seed]["epochs_run"],
           "best_epoch": lstm_info[main_seed]["best_epoch"], "holdout_metrics_main_seed": {k: float(v) for k, v in metrics(Yflat[te_h], lstm_hold[main_seed][te_h], sc_h[te_h]).items()},
           "history_main_seed": lstm_info[main_seed]["history"]}, open(f"{RES_DIR}/lstm_results.json", "w"), indent=1)
print("  LSTM SELESAI -> results/pred_LSTM.npz, lstm_*.csv/json")

## 4. Model Pembanding (Opsional)

Bagian ini menjalankan Ridge dan LightGBM sebagai model pembanding jika `COMPARISON = "full"`. Baseline sederhana seperti Naive dan Moving Average dihitung pada bagian Evaluasi.

In [ ]:
# ---- 4.0 Model pembanding: Ridge dan LightGBM ----
if COMPARISON == "full":
    try:
        import lightgbm as lgb
    except ImportError:
        import subprocess; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm"], check=True); import lightgbm as lgb
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import MinMaxScaler
    load_panel()
    LGB_SPACE = {"n_estimators": [200, 400, 800], "learning_rate": [0.03, 0.05, 0.1], "num_leaves": [15, 31, 63], "min_child_samples": [20, 50, 100],
                 "subsample": [0.7, 0.9, 1.0], "colsample_bytree": [0.5, 0.8, 1.0], "subsample_freq": [1]}
    if FAST_MODE: LGB_SPACE.update(n_estimators=[50, 100])
    print("\n[1/3] Tuning LightGBM (validasi berurutan waktu, data latih saja)...")
    LGB_T = cached("tune_lgbm", lambda: tune_sklearn(lgb.LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1), LGB_SPACE, LGB_ITER, "LightGBM"))

    def fit_pred_extra(train_end, test_ts):
        tr = np.where(TT < train_end)[0]; te = np.where(np.isin(TT, test_ts))[0]; out = {}
        sc = MinMaxScaler().fit(Xmain[tr]); rg = Ridge(alpha=1.0).fit(sc.transform(Xmain[tr]), Yflat[tr])
        lg = lgb.LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **LGB_T["best_params"]).fit(Xmain[tr], Yflat[tr])
        for nm, mdl, X in (("Ridge", rg, sc.transform(Xmain[te])), ("LightGBM", lg, Xmain[te])):
            out[nm] = np.full(len(D), np.nan); out[nm][te] = mdl.predict(X)
        return out
    print("\n[2/3] Holdout 80:20...")
    ph = fit_pred_extra(split_idx, list(range(split_idx, T)))
    print("\n[3/3] Rolling-origin...")
    pr = {"Ridge": np.full(len(D), np.nan), "LightGBM": np.full(len(D), np.nan)}
    for k, s in enumerate(starts):
        ts = list(range(s, s + ROLL_BLOCK)); te = np.where(np.isin(TT, ts))[0]; pk = fit_pred_extra(s, ts)
        for nm in pr: pr[nm][te] = pk[nm][te]
        print(f"      fold {k+1}/{N_FOLDS} selesai")
    for nm in ("Ridge", "LightGBM"): save_pred(nm, holdout=ph[nm], rolling=pr[nm])
    LGB_T["table"].to_csv(f"{RES_DIR}/lgbm_tuning.csv", index=False)
    json.dump({"model": "LightGBM", "best_params": {k: str(v) for k, v in LGB_T["best_params"].items()}, "search_space": {k: str(v) for k, v in LGB_SPACE.items()},
               "n_iter": LGB_ITER, "best_cv_mae": LGB_T["best_cv_mae"], "ridge_alpha": 1.0, "lightgbm_version": lgb.__version__}, open(f"{RES_DIR}/lgbm_results.json", "w"), indent=1)
    print("  MODEL PEMBANDING SELESAI -> results/pred_Ridge.npz, pred_LightGBM.npz")
else:
    print(f"COMPARISON='{COMPARISON}': Ridge dan LightGBM dilewati.")

## 5. Evaluasi

Mengevaluasi model pada baris pengujian yang sama menggunakan holdout 80:20 dan rolling-origin. Baseline sederhana dihitung dari fitur lag dan rolling yang tersedia.

In [ ]:
# ---- 5.0 Muat prediksi dan hitung baseline ----
load_panel()
PH, PR = {}, {}
for m in ["RF", "LSTM"]:
    f = f"{RES_DIR}/pred_{m}.npz"
    if not os.path.exists(f): continue
    z = np.load(f)
    for k in z.files:
        if k == "holdout": PH[m] = z[k]
        elif k == "rolling": PR[m] = z[k]
        elif k.startswith("holdout_s"): PH["LSTM_s" + k[9:]] = z[k]
        elif k.startswith("rolling_s"): PR["LSTM_s" + k[9:]] = z[k]

def baseline_vec(name, train_end):
    if name == "Naive": return D["lag_1"].values
    if name == "SeasonalNaive7": return D["lag_7"].values
    if name == "MA7": return D["rolling_mean_7d"].values
    if name == "MA30": return D["rolling_mean_30d"].values
    if name == "HistMean": return np.repeat(Y[:, :train_end].mean(1), T)
FOLD = np.full(len(D), -1)
for k, s in enumerate(starts): FOLD[(TT >= s) & (TT < s + ROLL_BLOCK)] = k
if COMPARISON in ("simple", "full"):
    for b in BASELINES:
        PH[b] = np.full(len(D), np.nan); PH[b][te_h] = baseline_vec(b, split_idx)[te_h]
        PR[b] = np.full(len(D), np.nan)
        for k, s in enumerate(starts):
            te = np.where(FOLD == k)[0]; PR[b][te] = baseline_vec(b, s)[te]
MODELS = model_list()
missing = [m for m in MODELS if m not in PH]
assert not missing, f"Prediksi belum ada untuk: {missing}. Jalankan section yang sesuai (atau ubah COMPARISON)."
SC_R = np.full(len(D), np.nan)
for k, s in enumerate(starts): te = np.where(FOLD == k)[0]; SC_R[te] = prod_scale(s)[te]
sc_h = prod_scale(split_idx)
print("Model dievaluasi:", MODELS)

In [ ]:
# ---- 5.1 Metrik holdout 80:20 ----
hold_tab = pd.DataFrame({m: metrics(Yflat[te_h], PH[m][te_h], sc_h[te_h]) for m in MODELS}).T.round(4)
hold_tab.to_csv(f"{RES_DIR}/holdout_metrics.csv")
print(f"=== HOLDOUT 80:20 ({len(te_h)} baris uji) ===\n", hold_tab)
seed_keys = [k for k in PH if k.startswith("LSTM_s")]
if seed_keys:
    sv_ = pd.DataFrame({k: metrics(Yflat[te_h], PH[k][te_h]) for k in seed_keys}).T
    print("\nVariasi seed LSTM (holdout):\n", sv_.round(4)); print("mean:", sv_.mean().round(3).to_dict(), "| std:", sv_.std().round(3).to_dict())

In [ ]:
# ---- 5.2 Uji signifikansi berpasangan (per produk) ----
def per_product_mae(p): return np.abs(Yflat[te_h] - p[te_h]).reshape(P, -1).mean(1)
rng = np.random.default_rng(SEED); sig_rows = []
for other in [m for m in ["LSTM", "LightGBM", "MA7", "Naive", "Ridge"] if m in PH]:
    a, b = per_product_mae(PH["RF"]), per_product_mae(PH[other]); dif = b - a
    bs = rng.choice(dif, size=(2000, len(dif))).mean(1)
    sig_rows.append({"comparison": f"RF vs {other}", "mean_MAE_diff(other-RF)": dif.mean(), "CI95_low": np.percentile(bs, 2.5),
                     "CI95_high": np.percentile(bs, 97.5), "wilcoxon_p": stats.wilcoxon(a, b).pvalue})
sig_tab = pd.DataFrame(sig_rows).round(5); sig_tab.to_csv(f"{RES_DIR}/holdout_significance.csv", index=False)
print("\n", sig_tab)

In [ ]:
# ---- 5.3 Rolling-origin ----
rows = []
for k in range(N_FOLDS):
    mk = np.where(FOLD == k)[0]
    for m in MODELS: rows.append({"fold": k + 1, "test_days": f"{starts[k]}-{starts[k]+ROLL_BLOCK-1}", "model": m, **metrics(Yflat[mk], PR[m][mk], SC_R[mk])})
roll_fold = pd.DataFrame(rows); roll_fold.round(4).to_csv(f"{RES_DIR}/rolling_by_fold.csv", index=False)
mk_all = np.where(FOLD >= 0)[0]
roll_pool = pd.DataFrame({m: metrics(Yflat[mk_all], PR[m][mk_all], SC_R[mk_all]) for m in MODELS}).T.round(4); roll_pool.to_csv(f"{RES_DIR}/rolling_pooled.csv")
roll_sum = roll_fold.groupby("model")[["MAE", "RMSE", "MAPE", "WAPE", "sMAPE", "MASE"]].agg(["mean", "std"]).loc[MODELS].round(3)
roll_sum.columns = [f"{a}_{b}" for a, b in roll_sum.columns]; roll_sum.to_csv(f"{RES_DIR}/rolling_summary_mean_std.csv")
print("=== ROLLING-ORIGIN (pooled) ===\n", roll_pool)
print("\nWAPE per fold:\n", roll_fold.pivot(index="model", columns="fold", values="WAPE").loc[MODELS].round(2))
print("\nRata-rata peringkat WAPE antar-fold (1 = terbaik):\n", roll_fold.pivot(index="fold", columns="model", values="WAPE")[MODELS].rank(axis=1).mean().round(2).sort_values().to_string())
roll_seed = pd.DataFrame({k: metrics(Yflat[mk_all], PR[k][mk_all], SC_R[mk_all]) for k in [k for k in PR if k.startswith("LSTM_s")]}).T.round(4)
if len(roll_seed):
    roll_seed.to_csv(f"{RES_DIR}/rolling_lstm_seeds.csv")
    print("\nSeed LSTM (rolling pooled):\n", roll_seed, "\nmean:", roll_seed.mean().round(3).to_dict(), "| std:", roll_seed.std().round(3).to_dict())

In [ ]:
# ---- 5.4 Hasil per kategori ----
def group_table(PRED, rows, by, models):
    gv = D[by].values[rows]; out = []
    for g in np.unique(gv):
        mk = rows[gv == g]
        for m in models:
            e = np.abs(Yflat[mk] - PRED[m][mk]); out.append({"group": g, "model": m, "WAPE": e.sum() / Yflat[mk].sum() * 100, "MAE": e.mean()})
    return pd.DataFrame(out).round(3)
GM = [m for m in ["MA7", "LightGBM", "RF", "LSTM"] if m in MODELS]
cat_h = group_table(PH, te_h, "category", GM); cat_r = group_table(PR, mk_all, "category", GM)
cat_h.to_csv(f"{RES_DIR}/holdout_by_category.csv", index=False); cat_r.to_csv(f"{RES_DIR}/rolling_by_category.csv", index=False)
print("WAPE holdout per kategori:\n", cat_h.pivot(index="group", columns="model", values="WAPE").round(2))
print("\n(Catatan: tidak ada breakdown per region karena satu produk-hari memuat beberapa region.)")

In [ ]:
# ---- 5.5 Simpan semua prediksi ----
np.savez_compressed(f"{RES_DIR}/predictions_all.npz", **{f"holdout__{m}": v for m, v in PH.items()}, **{f"rolling__{m}": v for m, v in PR.items()})
_pred = D[["product_id", "date", "category", "units_sold"]].copy(); _pred["fold"] = np.where(TT >= split_idx, "holdout", "")
for m in MODELS: _pred[f"holdout_{m}"] = PH[m]; _pred[f"rolling_{m}"] = PR[m]
_pred[(TT >= split_idx) | (FOLD >= 0)].to_csv(f"{RES_DIR}/predictions.csv", index=False)
print("  EVALUASI SELESAI -> results/holdout_metrics.csv, rolling_*.csv, predictions_all.npz")

In [ ]:
# ---- 5.6 Ringkasan RF vs LSTM (LSTM = rata-rata 3 seed) ----
# Menggunakan prediksi tersimpan dari bagian 5.0; tidak melakukan pelatihan ulang.
import matplotlib.pyplot as plt
SEEDS = sorted(k for k in PH if k.startswith("LSTM_s"))
assert SEEDS and "RF" in PH, "Prediksi per-seed LSTM / RF tidak ditemukan (cek results/pred_LSTM.npz dan pred_RF.npz)."
mk_all = np.where(FOLD >= 0)[0]
dates_all = pd.to_datetime(META["dates"])
print("Seed LSTM yang dipakai:", SEEDS)

# (a) Metrik holdout dan rolling-origin: RF (1 run) vs LSTM (mean/std/min/max antar seed)
def seed_summary(rows, PRED, scale, label):
    rf_m = pd.Series(metrics(Yflat[rows], PRED["RF"][rows], scale[rows]))
    ls_m = pd.DataFrame({k: metrics(Yflat[rows], PRED[k][rows], scale[rows]) for k in SEEDS}).T
    out = pd.DataFrame({"RF (1 run)": rf_m, "LSTM mean": ls_m.mean(), "LSTM std": ls_m.std(), "LSTM min": ls_m.min(), "LSTM max": ls_m.max()}).round(4)
    out.to_csv(f"{RES_DIR}/paper_{label}_rf_vs_lstm.csv"); return out
paper_hold = seed_summary(te_h, PH, sc_h, "holdout")
paper_roll = seed_summary(mk_all, PR, SC_R, "rolling_pooled")
print("=== HOLDOUT: RF vs LSTM (rata-rata seed) ===\n", paper_hold, "\n\n=== ROLLING-ORIGIN pooled ===\n", paper_roll)

# (b) Uji berpasangan per produk: tiap seed dan rata-rata seed (selisih MAE = LSTM - RF)
rf_pp = per_product_mae(PH["RF"])
cands = {k: per_product_mae(PH[k]) for k in SEEDS}
cands["LSTM seed-average"] = np.mean(list(cands.values()), axis=0)
sig_rows = []
for name, b in cands.items():
    dif = b - rf_pp; rng_ = np.random.default_rng(0)
    bs = [rng_.choice(dif, len(dif)).mean() for _ in range(5000)]
    sig_rows.append({"LSTM": name, "mean_MAE_diff(LSTM-RF)": dif.mean(), "CI95_low": np.percentile(bs, 2.5),
                     "CI95_high": np.percentile(bs, 97.5), "wilcoxon_p": stats.wilcoxon(dif).pvalue})
paper_sig = pd.DataFrame(sig_rows).round(4); paper_sig.to_csv(f"{RES_DIR}/paper_significance_rf_vs_lstm.csv", index=False)
print("\n=== UJI BERPASANGAN RF vs LSTM (holdout) ===\n", paper_sig)
ens_h = np.mean([PH[k] for k in SEEDS], axis=0)
print("WAPE holdout ensemble 3 seed (rata-rata prediksi):", round(metrics(Yflat[te_h], ens_h[te_h])["WAPE"], 3))

# (c) WAPE per blok rolling-origin: RF, tiap seed, rata-rata seed, ensemble, dan bias
fold_rows = []
for k in range(N_FOLDS):
    mk = np.where(FOLD == k)[0]; r = {"fold": k + 1, "start_date": str(dates_all[starts[k]])[:10], "RF": metrics(Yflat[mk], PR["RF"][mk])["WAPE"]}
    for s in SEEDS: r[s] = metrics(Yflat[mk], PR[s][mk])["WAPE"]
    r["LSTM mean"] = np.mean([r[s] for s in SEEDS]); r["LSTM ensemble"] = metrics(Yflat[mk], np.mean([PR[s][mk] for s in SEEDS], axis=0))["WAPE"]
    r["bias_RF"] = (PR["RF"][mk] - Yflat[mk]).mean()
    for s in SEEDS: r[f"bias_{s}"] = (PR[s][mk] - Yflat[mk]).mean()
    fold_rows.append(r)
paper_fold = pd.DataFrame(fold_rows).round(2); paper_fold.to_csv(f"{RES_DIR}/paper_rolling_by_fold.csv", index=False)
print("\n=== WAPE PER BLOK ROLLING-ORIGIN ===\n", paper_fold.to_string(index=False))
print("RF per blok: mean", round(paper_fold["RF"].mean(), 2), "| std", round(paper_fold["RF"].std(), 2),
      "| LSTM mean-seed per blok: mean", round(paper_fold["LSTM mean"].mean(), 2), "| std", round(paper_fold["LSTM mean"].std(), 2))
print("WAPE rolling ensemble 3 seed (pooled):", round(metrics(Yflat[mk_all], np.mean([PR[k] for k in SEEDS], axis=0)[mk_all])["WAPE"], 3))

# (d) WAPE per kategori (holdout): RF vs rata-rata seed LSTM
cat_v = D["category"].values[te_h]; cat_rows = []
for g in np.unique(cat_v):
    mk = te_h[cat_v == g]
    cat_rows.append({"category": g, "RF": metrics(Yflat[mk], PH["RF"][mk])["WAPE"],
                     "LSTM mean": np.mean([metrics(Yflat[mk], PH[s][mk])["WAPE"] for s in SEEDS])})
paper_cat = pd.DataFrame(cat_rows).round(2); paper_cat.to_csv(f"{RES_DIR}/paper_holdout_by_category.csv", index=False)
print("\n=== WAPE PER KATEGORI (holdout) ===\n", paper_cat.to_string(index=False))

# (e) Gambar untuk paper: WAPE per blok, RF vs tiap seed LSTM (tanpa model lain)
fig, ax = plt.subplots(figsize=(6.2, 3.0), dpi=300); xs = paper_fold["start_date"].str[5:].values
for s, ls_ in zip(SEEDS, ["-", "--", ":"]): ax.plot(xs, paper_fold[s], ls_, marker="o", ms=4, color="0.55", lw=1.3, label=f"LSTM {s[5:]}".replace("s", "seed "))
ax.plot(xs, paper_fold["RF"], "-", marker="o", color="#1f5fbf", lw=2.2, ms=5, label="RF")
ax.set_ylabel("WAPE (%)"); ax.set_xlabel("First day of 7-day test block (month-day, 2026)"); ax.grid(axis="y", lw=.4, color="0.85")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/fig05e_paper_rolling_rf_vs_lstm_seeds.png"); plt.close(fig)
print("\nTersimpan: results/paper_*.csv dan figures/fig05e_paper_rolling_rf_vs_lstm_seeds.png")

## 6. Rekomendasi Restock

Menghitung rekomendasi reorder point pada data pengujian dan menjalankan simulasi inventori periodic-review untuk mengevaluasi fill rate serta biaya berdasarkan kualitas prediksi.

In [ ]:
# ---- 6.0 Muat prediksi ----
load_panel(); PH, PR = load_all_preds()
LEAD_TIME, REVIEW, Z = BASE_L, BASE_R, BASE_Z
print(f"Asumsi: lead time {LEAD_TIME} hari, review period {REVIEW} hari, z={Z}, sigma = std permintaan harian produk pada periode latih, "
      f"posisi stok = inventory_level akhir hari sebelumnya (tidak ada data pesanan outstanding)")

In [ ]:
# ---- 6.1 A. Rekomendasi reorder point (holdout) ----
print("\n[1/3] Rekomendasi restock per baris uji (ROP = ramalan x lead time + z x sigma x sqrt(LT+R))...")
hist_std_rows = np.repeat(Y[:, :split_idx].std(1), T)[te_h]
inv_rows = D["inventory_lag1"].values[te_h]
def recommend(pred):
    pred = np.clip(pred, 0, None); ltd = pred * LEAD_TIME; ss = Z * hist_std_rows * np.sqrt(LEAD_TIME + REVIEW); rop = ltd + ss
    qty = np.clip(rop - inv_rows, 0, None); dos = np.where(pred > 0, inv_rows / np.maximum(pred, 1e-9), 999.0)
    prio = np.where(dos < LEAD_TIME, "CRITICAL", np.where(dos < 2 * LEAD_TIME, "HIGH", np.where(qty > 0, "MEDIUM", "OK")))
    return pd.DataFrame({"predicted_daily_demand": pred, "lead_time_demand": ltd, "safety_stock": ss, "reorder_point": rop, "current_inventory": inv_rows,
                         "restock_quantity": qty, "needs_restock": qty > 0, "days_of_supply": dos, "priority": prio})
rec_rows = []
for m in [m for m in ["RF", "LSTM"] if m in PH]:
    r = recommend(PH[m][te_h]); pr_ = r["priority"].value_counts(normalize=True) * 100
    rec_rows.append({"model": m, "needs_restock_%": 100 * r["needs_restock"].mean(), "total_restock_units": r["restock_quantity"].sum(),
                     **{f"priority_{p}_%": pr_.get(p, 0.0) for p in ["CRITICAL", "HIGH", "MEDIUM", "OK"]}})
    if m == "RF":
        out = pd.concat([D.loc[te_h, ["date", "product_id", "category", "units_sold"]].reset_index(drop=True), r], axis=1)
        out.round(2).to_csv(f"{RES_DIR}/restock_recommendations_RF.csv", index=False)
REC = pd.DataFrame(rec_rows).round(2); REC.to_csv(f"{RES_DIR}/restock_recommendation_summary.csv", index=False)
print(REC.to_string(index=False))
print("Catatan: persentase 'needs_restock' saja BUKAN bukti efektivitas; evaluasi efektivitas ada di simulasi (bagian B).")

In [ ]:
# ---- 6.2 B. Simulasi inventori - definisi kebijakan ----
print("\n[2/3] Simulasi inventori periodic-review...")
# Kebijakan: tinjau tiap R hari; order-up-to S = laju*(L+R) + z*sigma*sqrt(L+R); Q = max(0, S - IP);
# IP = stok di tangan + pesanan outstanding (tiba L hari setelah dipesan); permintaan tak terpenuhi = lost sales;
# permintaan simulasi = units_sold aktual. Stok awal identik untuk semua kebijakan.
ORACLE = "Oracle (perfect lead-time demand)"
sim_t0 = starts[0]; Ys = Y[:, sim_t0:]; H = Ys.shape[1]; d0 = ROLL_BLOCK
pre = Y[:, :starts[1]]                                         # informasi yang tersedia saat simulasi dimulai
dmean, dstd = pre.mean(1), pre.std(1) + 1e-6
price_p = D.loc[D["t"] < starts[1]].groupby("product_id")["effective_price"].mean().reindex(D["product_id"].unique()).values

def F_of(m): return PR[m].reshape(P, T)[:, sim_t0:]
def smooth(F, k=7): return np.stack([F[:, max(0, j - k + 1):j + 1].mean(1) for j in range(F.shape[1])], 1)   # rata-rata ramalan harian terkini
def oracle_rate(L, R): return np.stack([Ys[:, j:j + L + R].mean(1) for j in range(H)], 1)
POL = {"Static mean (s,S)": np.repeat(dmean[:, None], H, 1)}
for nm in ["Naive", "SeasonalNaive7", "MA7", "LightGBM", "RF", "LSTM"]:
    if nm in PR: POL[nm] = F_of(nm)
for nm in ["LightGBM", "RF", "LSTM"]:
    if nm in POL: POL[f"{nm} (7d-smoothed)"] = smooth(POL[nm])
for k_, v_ in POL.items(): assert not np.isnan(v_).any(), f"NaN pada ramalan {k_}"

def make_sigma(F, mode, mult):
    S = np.zeros((P, H))
    if mode == "demand_std": S[:] = dstd[:, None] * mult
    else:                                                       # sigma dari RMSE ramalan blok sebelumnya (skala per produk)
        for b in range(1, H // ROLL_BLOCK):
            e = Ys[:, (b - 1) * ROLL_BLOCK:b * ROLL_BLOCK] - F[:, (b - 1) * ROLL_BLOCK:b * ROLL_BLOCK]
            S[:, b * ROLL_BLOCK:(b + 1) * ROLL_BLOCK] = np.sqrt((e ** 2).mean()) * (dmean / dmean.mean())[:, None] * mult
    return S

def simulate(F, SIG, L, R, z):
    hold_u = HOLD_ANNUAL / 365 * price_p; short_u = MARGIN * price_p
    oh = dmean * (L + R) + z * dstd * np.sqrt(L + R)
    arr = np.zeros((P, H + L + 2)); lost = np.zeros(P); dem = np.zeros(P); so = 0.0; inv = np.zeros(P)
    hold = np.zeros(P); nord = np.zeros(P); uord = np.zeros(P); Sl = []
    for d_ in range(d0, H):
        oh = oh + arr[:, d_]
        if (d_ - d0) % R == 0:
            ip = oh + arr[:, d_ + 1:].sum(1)
            S = F[:, d_] * (L + R) + z * SIG[:, d_] * np.sqrt(L + R)
            q = np.maximum(0, S - ip); arr[:, d_ + L] += q; nord += q > 0; uord += q; Sl.append(S)
        sales = np.minimum(oh, Ys[:, d_]); l = Ys[:, d_] - sales; oh = oh - sales
        lost += l; dem += Ys[:, d_]; so += (l > 1e-9).sum(); inv += oh; hold += oh * hold_u
    nd = H - d0
    r = dict(fill_rate=100 * (1 - lost.sum() / dem.sum()), stockout_rate=100 * so / (P * nd), avg_on_hand=inv.sum() / (P * nd), holding_cost=hold.sum(),
             shortage_cost=(lost * short_u).sum(), ordering_cost=nord.sum() * ORDER_COST, n_orders=nord.sum(), units_ordered=uord.sum())
    r["total_cost"] = r["holding_cost"] + r["shortage_cost"] + r["ordering_cost"]
    return r, np.array(Sl)

def run_all(mode, L=BASE_L, R=BASE_R, z=BASE_Z, mult=1.0):
    out, Ss = {}, {}
    for nm, F in {ORACLE: oracle_rate(L, R), **POL}.items():
        out[nm], Ss[nm] = simulate(F, make_sigma(F, mode, mult), L, R, z)
    for nm in out: out[nm]["S_MAE_vs_oracle"] = float(np.abs(Ss[nm] - Ss[ORACLE]).mean())
    return out

In [ ]:
# ---- 6.3 B. Hasil simulasi, sensitivitas, dan injeksi error ----
base_rows = [{"sigma_mode": mode, "policy": nm, **r} for mode in ["demand_std", "forecast_rmse"] for nm, r in run_all(mode).items()]
INV_BASE = pd.DataFrame(base_rows); INV_BASE.round(2).to_csv(f"{RES_DIR}/inventory_baseline.csv", index=False)
print(f"\n=== SIMULASI INVENTORI (hari {sim_t0+d0}-{T-1}, L={BASE_L}, R={BASE_R}, z={BASE_Z}) ===")
print(INV_BASE.round(2).to_string(index=False))

sens = []
for mode in ["demand_std", "forecast_rmse"]:
    for z_ in [1.28, 1.65, 2.33]:
        for nm, r in run_all(mode, z=z_).items(): sens.append({"sigma_mode": mode, "factor": "service level z", "value": z_, "policy": nm, **r})
    for L_ in [3, 5, 7, 10]:
        for nm, r in run_all(mode, L=L_).items(): sens.append({"sigma_mode": mode, "factor": "lead time (days)", "value": L_, "policy": nm, **r})
    for m_ in [0.5, 1.0, 1.5]:
        for nm, r in run_all(mode, mult=m_).items(): sens.append({"sigma_mode": mode, "factor": "sigma multiplier", "value": m_, "policy": nm, **r})
    for R_ in [1, 7]:
        for nm, r in run_all(mode, R=R_).items(): sens.append({"sigma_mode": mode, "factor": "review period (days)", "value": R_, "policy": nm, **r})
SENS = pd.DataFrame(sens); SENS.round(3).to_csv(f"{RES_DIR}/inventory_sensitivity.csv", index=False)

# diagnostik: seberapa dekat laju permintaan tiap kebijakan dengan permintaan nyata L+R hari ke depan
OR_ = oracle_rate(BASE_L, BASE_R); ds_ = list(range(d0, H - (BASE_L + BASE_R) + 1)); diag = []
true_rate = np.stack([Ys[:, j:j + BASE_L + BASE_R].mean(1) for j in ds_], 1)
for nm, F in POL.items():
    e = np.stack([F[:, j] for j in ds_], 1) - true_rate
    diag.append({"policy": nm, "rate_MAE": np.abs(e).mean(), "rate_bias": e.mean(), "rate_WAPE_%": 100 * np.abs(e).sum() / true_rate.sum()})
DIAG = pd.DataFrame(diag).round(3); DIAG.to_csv(f"{RES_DIR}/inventory_rate_diagnostic.csv", index=False)
print("\nDiagnostik laju permintaan lead-time:\n", DIAG.to_string(index=False))

# injeksi error: laju sempurna + noise Gaussian multiplikatif (dampak error forecast terhadap keputusan)
inj = []
for lvl in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    for mode in ["demand_std", "forecast_rmse"]:
        reps = []
        for sd_ in range(3):
            g = np.random.default_rng(SEED + sd_); Fn = np.clip(OR_ * (1 + g.normal(0, lvl, OR_.shape)), 0, None)
            r, _ = simulate(Fn, make_sigma(Fn, mode, 1.0), BASE_L, BASE_R, BASE_Z)
            r["WAPE_of_forecast"] = 100 * np.abs(Fn - OR_).sum() / OR_.sum(); reps.append(r)
        inj.append({"noise_sd": lvl, "sigma_mode": mode, **pd.DataFrame(reps).mean().to_dict()})
INJ = pd.DataFrame(inj); INJ.round(3).to_csv(f"{RES_DIR}/inventory_error_injection.csv", index=False)
print("\nInjeksi error:\n", INJ[["noise_sd", "sigma_mode", "WAPE_of_forecast", "fill_rate", "stockout_rate", "total_cost"]].round(2).to_string(index=False))
dec = INV_BASE[INV_BASE["sigma_mode"] == "forecast_rmse"][["policy", "n_orders", "units_ordered", "S_MAE_vs_oracle"]].copy()
dec["order_share_%"] = 100 * dec["n_orders"] / (P * len(range(d0, H, BASE_R))); dec.round(2).to_csv(f"{RES_DIR}/inventory_decisions.csv", index=False)
print("\n[3/3] RESTOCK SELESAI -> results/restock_*.csv, inventory_*.csv")

## 7. Visualisasi

Membuat visualisasi eksplorasi data, evaluasi model, feature importance, dan simulasi inventori. Setiap gambar disimpan sebagai berkas PNG pada folder `figures/`.

In [ ]:
# ---- 7.0 Muat hasil dan pengaturan gaya ----
load_panel(); PH, PR = load_all_preds(); MODELS = [m for m in model_list() if m in PH]
plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.facecolor": "white"})
COL = {"Naive": "#9CA3AF", "SeasonalNaive7": "#6B7280", "MA7": "#F59E0B", "MA30": "#D97706", "HistMean": "#D1D5DB", "Ridge": "#10B981",
       "LightGBM": "#7C3AED", "RF": "#2563EB", "LSTM": "#DC2626", "Actual": "#111827"}
def sv(fig, name):
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/{name}.png"); plt.close(fig); print("tersimpan:", name)
rd = lambda f, **kw: pd.read_csv(f"{RES_DIR}/{f}.csv", **kw)
dates_ = pd.to_datetime(META["dates"])

In [ ]:
# ---- 7.1 Fig 1 - EDA (distribusi, tren harian, per kategori) ----
fig, ax = plt.subplots(figsize=(6.5, 4.5)); ax.hist(Yflat, bins=40, color=COL["RF"], alpha=.8, edgecolor="white")
ax.axvline(Yflat.mean(), color="red", ls="--", label=f"Mean = {Yflat.mean():.1f}"); ax.axvline(np.median(Yflat), color="orange", ls=":", label=f"Median = {np.median(Yflat):.1f}")
ax.set_xlabel("Daily units sold per product"); ax.set_ylabel("Frequency"); ax.legend(); sv(fig, "fig01a_demand_distribution")

dm = Y.mean(0); fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(dates_[:split_idx], dm[:split_idx], color=COL["Actual"], label="Training period"); ax.plot(dates_[split_idx - 1:], dm[split_idx - 1:], color=COL["RF"], ls="--", label="Test period")
ax.axvline(dates_[split_idx], color="gray", ls=":"); ax.set_xlabel("Date"); ax.set_ylabel("Average units sold per product"); ax.legend(); fig.autofmt_xdate(); sv(fig, "fig01b_daily_average_demand")

cm = D.groupby("category")["units_sold"].mean().sort_values(); fig, ax = plt.subplots(figsize=(6.5, 4.5)); b = ax.barh(cm.index, cm.values, color=COL["RF"], alpha=.85)
for r_, v in zip(b, cm.values): ax.text(v + .1, r_.get_y() + r_.get_height() / 2, f"{v:.1f}", va="center", fontsize=9)
ax.set_xlabel("Average units sold per product-day"); sv(fig, "fig01c_demand_by_category")

In [ ]:
# ---- 7.2 Fig 2 - Feature importance dan ablasi ----
gi = rd("rf_gini_importance", index_col=0)["gini"].head(15).sort_values(); fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(gi.index, gi.values, color=COL["RF"], alpha=.85)
ax.set_xlabel("Gini importance (Random Forest)"); sv(fig, "fig02a_gini_importance_top15")
p_ = rd("rf_perm_importance_group", index_col=0).sort_values("MAE_increase"); fig, ax = plt.subplots(figsize=(6.5, 4)); ax.barh(p_.index, p_["MAE_increase"], xerr=p_["std"], color=COL["RF"], alpha=.85)
ax.set_xlabel("Increase in test MAE when the group is permuted"); sv(fig, "fig02b_permutation_importance_groups")
p_ = rd("rf_perm_importance_feature", index_col=0).sort_values("MAE_increase"); fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(p_.index, p_["MAE_increase"], xerr=p_["std"], color=COL["RF"], alpha=.85)
ax.set_xlabel("Increase in test MAE when the feature is permuted"); sv(fig, "fig02c_permutation_importance_features")
cr = rd("corr_rolling_lag", index_col=0); fig, ax = plt.subplots(figsize=(7.5, 6.5)); im = ax.imshow(cr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cr))); ax.set_yticks(range(len(cr))); ax.set_xticklabels(cr.columns, rotation=90, fontsize=8); ax.set_yticklabels(cr.index, fontsize=8)
fig.colorbar(im, ax=ax, shrink=.8, label="Pearson correlation"); sv(fig, "fig02d_correlation_rolling_lag")
ab = rd("rf_ablation").sort_values("MAE"); fig, ax = plt.subplots(figsize=(8, 4.8)); ax.barh(ab["feature_set"], ab["MAE"], color=COL["RF"], alpha=.85)
ax.set_xlabel("Holdout MAE of Random Forest"); sv(fig, "fig02e_ablation_rf")

In [ ]:
# ---- 7.3 Fig 3 - Aktual vs prediksi per kategori ----
pids = list(D["product_id"].unique())
for cat in [c for c in ["Shoes", "Electronics", "Groceries", "Beauty"] if c in set(D["category"])]:
    pid = D.loc[D["category"] == cat, "product_id"].iloc[0]; pi = pids.index(pid); ix = np.arange(pi * T, (pi + 1) * T)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(dates_[split_idx - 14:], Y[pi, split_idx - 14:], color=COL["Actual"], lw=2, label="Actual")
    for m in [m for m in ["MA7", "LightGBM", "RF", "LSTM"] if m in PH]:
        ax.plot(dates_[split_idx:], PH[m][ix][split_idx:], color=COL[m], ls="--", lw=1.6, marker="o", ms=3, label=m)
    ax.axvline(dates_[split_idx], color="gray", ls=":"); ax.set_xlabel("Date"); ax.set_ylabel("Units sold"); ax.set_title(f"{cat} ({pid})"); ax.legend(ncol=3, fontsize=9)
    fig.autofmt_xdate(); sv(fig, f"fig03_forecast_{cat.lower()}")

In [ ]:
# ---- 7.4 Fig 4 - Scatter aktual vs prediksi ----
for m in [m for m in ["RF", "LSTM", "LightGBM"] if m in PH]:
    y_, p_ = Yflat[te_h], PH[m][te_h]; fig, ax = plt.subplots(figsize=(5.5, 5)); ax.scatter(y_, p_, s=6, alpha=.25, color=COL[m])
    lim = [min(y_.min(), p_.min()), max(y_.max(), p_.max())]; ax.plot(lim, lim, "k--", lw=1, label="Perfect fit"); mm = metrics(y_, p_)
    ax.set_title(f"{m}: MAE={mm['MAE']:.2f}, WAPE={mm['WAPE']:.1f}%"); ax.set_xlabel("Actual units sold"); ax.set_ylabel("Predicted units sold"); ax.legend(); sv(fig, f"fig04_scatter_{m.lower()}")

In [ ]:
# ---- 7.5 Fig 5 - Perbandingan model (holdout, rolling-origin, kategori, LSTM) ----
ht = rd("holdout_metrics", index_col=0); order = ht["WAPE"].sort_values().index
fig, ax = plt.subplots(figsize=(7.5, 4.5)); ax.bar(order, ht.loc[order, "WAPE"], color=[COL[m] for m in order])
for i, m in enumerate(order): ax.text(i, ht.loc[m, "WAPE"] + .2, f"{ht.loc[m,'WAPE']:.1f}", ha="center", fontsize=9)
ax.set_ylabel("WAPE (%) on holdout test set"); plt.setp(ax.get_xticklabels(), rotation=30, ha="right"); sv(fig, "fig05a_holdout_wape_all_models")
rf_ = rd("rolling_by_fold"); fig, ax = plt.subplots(figsize=(7.5, 4.5))
for m in [m for m in ["Naive", "MA7", "LightGBM", "RF", "LSTM"] if m in set(rf_["model"])]:
    s_ = rf_[rf_.model == m].sort_values("fold"); ax.plot(s_["fold"], s_["WAPE"], marker="o", color=COL[m], label=m)
ax.set_xlabel("Rolling-origin fold"); ax.set_ylabel("WAPE (%)"); ax.set_xticks(sorted(rf_["fold"].unique())); ax.legend(); sv(fig, "fig05b_rolling_origin_wape")
ch = rd("holdout_by_category").pivot(index="group", columns="model", values="WAPE"); ms = [m for m in ["MA7", "LightGBM", "RF", "LSTM"] if m in ch.columns]
fig, ax = plt.subplots(figsize=(8, 4.5)); x = np.arange(len(ch)); bw = .8 / len(ms)
for i, m in enumerate(ms): ax.bar(x + (i - (len(ms) - 1) / 2) * bw, ch[m].values, bw, color=COL[m], label=m)
ax.set_xticks(x); ax.set_xticklabels(ch.index, rotation=30, ha="right"); ax.set_ylabel("WAPE (%) on holdout"); ax.legend(ncol=len(ms)); sv(fig, "fig05c_wape_by_category")
lj = json.load(open(f"{RES_DIR}/lstm_results.json")); h_ = lj["history_main_seed"]; fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(range(1, len(h_["loss"]) + 1), h_["loss"], color=COL["LSTM"], label="Training loss"); ax.plot(range(1, len(h_["val_loss"]) + 1), h_["val_loss"], color=COL["Actual"], ls="--", label="Validation loss")
ax.axvline(lj["best_epoch"], color="gray", ls=":", label=f"Best epoch = {lj['best_epoch']}"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (scaled target)"); ax.legend(); sv(fig, "fig05d_lstm_training_curve")

In [ ]:
# ---- 7.6 Fig 6 - Restock dan simulasi inventori ----
ib = rd("inventory_baseline"); SENS = rd("inventory_sensitivity"); INJ = rd("inventory_error_injection")
for mode in ["demand_std", "forecast_rmse"]:
    t_ = ib[ib.sigma_mode == mode].set_index("policy")[["holding_cost", "shortage_cost", "ordering_cost"]]
    fig, ax = plt.subplots(figsize=(8.5, 5)); bot = np.zeros(len(t_))
    for c, colr in zip(t_.columns, ["#60A5FA", "#F87171", "#FBBF24"]): ax.bar(t_.index, t_[c], bottom=bot, color=colr, label=c.replace("_", " ")); bot += t_[c].values
    ax.set_ylabel("Total cost (currency units)"); plt.setp(ax.get_xticklabels(), rotation=35, ha="right"); ax.legend(); sv(fig, f"fig06a_inventory_cost_{mode}")
    t2 = ib[ib.sigma_mode == mode].set_index("policy")["fill_rate"]; fig, ax = plt.subplots(figsize=(8.5, 4.8)); ax.bar(t2.index, t2.values, color="#34D399")
    ax.set_ylim(max(0, t2.min() - 5), 100.5); ax.set_ylabel("Fill rate (%)"); plt.setp(ax.get_xticklabels(), rotation=35, ha="right"); sv(fig, f"fig06b_fill_rate_{mode}")
ORACLE = [p for p in ib.policy.unique() if p.startswith("Oracle")][0]
PC = {ORACLE: "#111827", "Static mean (s,S)": "#9CA3AF", "MA7": COL["MA7"], "LightGBM": COL["LightGBM"], "RF": COL["RF"], "LSTM": COL["LSTM"], "RF (7d-smoothed)": "#60A5FA"}
PLOT_POL = [p for p in PC if p in set(SENS.policy)]
for fac, fn in [("service level z", "service"), ("lead time (days)", "leadtime"), ("sigma multiplier", "sigma")]:
    for metric, lab in [("total_cost", "Total cost"), ("fill_rate", "Fill rate (%)")]:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        for pol in PLOT_POL:
            s_ = SENS[(SENS.sigma_mode == "forecast_rmse") & (SENS.factor == fac) & (SENS.policy == pol)]; ax.plot(s_["value"], s_[metric], marker="o", color=PC[pol], label=pol)
        ax.set_xlabel(fac); ax.set_ylabel(lab); ax.legend(fontsize=8); sv(fig, f"fig06c_sens_{fn}_{metric}")
for metric, lab in [("total_cost", "Total cost"), ("fill_rate", "Fill rate (%)")]:
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for mode, ls in [("demand_std", "--"), ("forecast_rmse", "-")]:
        s_ = INJ[INJ.sigma_mode == mode]; ax.plot(s_["WAPE_of_forecast"], s_[metric], marker="o", ls=ls, label=f"sigma: {mode}")
    ax.set_xlabel("Forecast WAPE (%) (injected noise)"); ax.set_ylabel(lab); ax.legend(); sv(fig, f"fig06d_error_injection_{metric}")
rs = rd("restock_recommendation_summary"); pc_ = ["priority_CRITICAL_%", "priority_HIGH_%", "priority_MEDIUM_%", "priority_OK_%"]
fig, ax = plt.subplots(figsize=(6.5, 4.5)); bot = np.zeros(len(rs))
for c, colr in zip(pc_, ["#DC2626", "#F59E0B", "#FBBF24", "#34D399"]): ax.bar(rs["model"], rs[c], bottom=bot, color=colr, label=c.split("_")[1]); bot += rs[c].values
ax.set_ylabel("Share of test records (%)"); ax.legend(title="Priority"); sv(fig, "fig06e_restock_priority_share")
print(f"\n  VISUALISASI SELESAI - {len(os.listdir(FIG_DIR))} file di '{FIG_DIR}/'")

## 8. Ekspor Hasil

Menyimpan konfigurasi eksperimen, ringkasan metrik, tabel, dan gambar ke folder keluaran serta mengemas hasil penelitian ke dalam berkas ZIP.

In [ ]:
# ---- 8.0 Konfigurasi, ringkasan, dan zip ----
import tensorflow as tf
try: import lightgbm as lgb; lgb_v = lgb.__version__
except Exception: lgb_v = None
env = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "tensorflow": tf.__version__, "lightgbm": lgb_v,
       "seed": SEED, "FAST_MODE": FAST_MODE, "feature_set": FEATURE_SET, "comparison": COMPARISON, "split": f"{int(TRAIN_RATIO*100)}:{100-int(TRAIN_RATIO*100)} time-based",
       "rolling_origin": {"folds": N_FOLDS, "block_days": ROLL_BLOCK}, "sim_assumptions": dict(HOLD_ANNUAL=HOLD_ANNUAL, MARGIN=MARGIN, ORDER_COST=ORDER_COST, L=BASE_L, R=BASE_R, z=BASE_Z),
       "runtime_min": round((time.time() - T_START) / 60, 1)}
for nm in ("rf", "lstm", "lgbm"):
    f = f"{RES_DIR}/{nm}_results.json"
    if os.path.exists(f): env[nm] = {k: v for k, v in json.load(open(f)).items() if k != "history_main_seed"}
json.dump(env, open(f"{RES_DIR}/config_and_hyperparameters.json", "w"), indent=1, default=str)

rd = lambda f, **kw: pd.read_csv(f"{RES_DIR}/{f}.csv", **kw)
print("=" * 28 + " RINGKASAN HASIL EKSPERIMEN " + "=" * 28)
print("FEATURE_SET:", FEATURE_SET, "| COMPARISON:", COMPARISON, "| FAST_MODE:", FAST_MODE)
for t_, f_, kw in [("HOLDOUT", "holdout_metrics", {"index_col": 0}), ("ROLLING pooled", "rolling_pooled", {"index_col": 0}), ("ROLLING mean/std", "rolling_summary_mean_std", {"index_col": 0}),
                   ("SIGNIFICANCE", "holdout_significance", {}), ("ABLATION", "rf_ablation", {}), ("PERM GROUP", "rf_perm_importance_group", {"index_col": 0}),
                   ("RESTOCK RECOMMENDATION", "restock_recommendation_summary", {}), ("INVENTORY baseline", "inventory_baseline", {}), ("RATE DIAGNOSTIC", "inventory_rate_diagnostic", {})]:
    if os.path.exists(f"{RES_DIR}/{f_}.csv"): print(f"\n{t_}:\n", rd(f_, **kw).round(3).to_string())
for f_ in ("lstm_seed_variability", "rolling_lstm_seeds"):
    if os.path.exists(f"{RES_DIR}/{f_}.csv"): print(f"\n{f_}:\n", rd(f_).round(4).to_string())
print("\nHYPERPARAMS:", json.dumps({k: env[k].get("best_params", env[k].get("best_cfg")) for k in ("rf", "lstm", "lgbm") if k in env}, default=str), "| runtime (menit):", env["runtime_min"])

zp = "jaic_revision_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z_:
    for folder in (RES_DIR, FIG_DIR):
        for root, _, fs in os.walk(folder):
            if "cache" in root: continue
            for f_ in fs:
                if f_ == "predictions_all.npz": continue
                z_.write(os.path.join(root, f_))
    for f_ in ("metadata.json", "label_maps.json"): z_.write(f"{DATA_DIR}/{f_}")
print(f"\nZip dibuat: {zp} ({os.path.getsize(zp)/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(zp)
except Exception:
    pass